# AI-Based Predictive Maintenance for Mining Equipment

This is the **one working notebook** for the capstone. We append each new equipment result here. The model code remains in `Models/`, detailed metrics in `Results/`, and figures in `Figures/`. The notebook reads those saved results; **Run All does not download data or train models**.

The task is classifying a component's **current fault condition** from sensor and visual data. Most sources are laboratory analogues, not images and signals from actual mining machines.

## Results so far

In VS Code, select the **Capstone (.venv)** kernel and run the next cell. **Strict** means whole recordings, bearings, or speeds were held out and is the main test. **Time split** reuses each recording across train and test and is easier; it cannot replace the strict score. A result counts toward the professor's >80% target only if **both strict scores** exceed 80%.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'Results').is_dir())
equipment = [  # Append future equipment here when both models have saved results.
    ('Pump', 'pump'),
    ('Haul Truck', 'haul_truck'),
    ('Belt Conveyor', 'belt_conveyor'),
    ('Excavator', 'excavator'),
    ('Dozer', 'dozer'),
    ('Wheel Loader', 'wheel_loader'),
    ('Drill Rig', 'drill_rig'),
    ('Grader', 'grader'),
]

def read_result(machine, suffix):
    path = root / 'Results' / machine / f'{machine}_{suffix}.json'
    return json.loads(path.read_text()) if path.exists() else {}

def pct(score):
    return round(score * 100, 2) if score is not None else None

rows = []
for name, machine in equipment:
    sensor = read_result(machine, 'csv_4class').get('validation_selected', {}).get('score', {})
    visual = read_result(machine, 'cnn_pretrained_4class').get('recording_level', {})
    sensor_time = read_result(machine, 'csv_4class_timesplit').get('models', {}).get('Random forest', {})
    visual_time = read_result(machine, 'cnn_pretrained_4class_timesplit').get('recording_level', {})
    s, v = sensor.get('accuracy'), visual.get('accuracy')
    rows.append({'Equipment': name, 'Strict sensor %': pct(s), 'Strict visual %': pct(v),
                 'Time sensor %': pct(sensor_time.get('accuracy')),
                 'Time visual %': pct(visual_time.get('accuracy')),
                 'Status': 'Both >80%' if s is not None and v is not None and s > .8 and v > .8
                           else 'Evaluated' if s is not None and v is not None else 'Pending'})
display(pd.DataFrame(rows).set_index('Equipment'))

,Strict sensor %,Strict visual %,Time sensor %,Time visual %,Status
Equipment,,,,,
Pump,95.83,100.00,NaN,NaN,Both >80%
Haul Truck,80.00,41.67,100.00,91.30,Evaluated
Belt Conveyor,62.50,56.25,100.00,81.25,Evaluated
Excavator,18.75,31.25,93.75,93.75,Evaluated
Dozer,62.50,56.25,100.00,100.00,Evaluated
Wheel Loader,31.25,31.25,93.75,81.25,Evaluated
Drill Rig,NaN,NaN,NaN,NaN,Pending
Grader,NaN,NaN,NaN,NaN,Pending


## Evidence by equipment

- **Pump — support bearing:** [sensor and visual comparison](../Results/pump/pump_sensor_visual_comparison.md) · [sensor figure](../Figures/pump/pump_csv_4class.png) · [visual figure](../Figures/pump/pump_cnn_pretrained_4class.png)
- **Haul Truck — wheel-motor bearing analogue:** [comparison](../Results/haul_truck/haul_truck_sensor_visual_comparison.md) · [sensor figure](../Figures/haul_truck/haul_truck_csv_4class.png) · [visual figure](../Figures/haul_truck/haul_truck_cnn_pretrained_4class.png)
- **Belt Conveyor — geared-drive analogue:** [comparison](../Results/belt_conveyor/belt_conveyor_sensor_visual_comparison.md) · [sensor figure](../Figures/belt_conveyor/belt_conveyor_csv_4class.png) · [visual figure](../Figures/belt_conveyor/belt_conveyor_cnn_pretrained_4class.png)
- **Excavator — swing-drive analogue:** [sensor result](../Results/excavator/excavator_csv_4class.json) · [visual result](../Results/excavator/excavator_cnn_pretrained_4class.json)
- **Dozer — final-drive analogue:** [method note](../Documentation/week_06_dozer.md) · [sensor result](../Results/dozer/dozer_csv_4class.json) · [visual result](../Results/dozer/dozer_cnn_pretrained_4class.json)
- **Wheel Loader — electric drive motor analogue:** [Week 7 report](../Documentation/week_07_wheel_loader.md) · [sensor result](../Results/wheel_loader/wheel_loader_csv_4class.json) · [visual result](../Results/wheel_loader/wheel_loader_cnn_pretrained_4class.json)
- **Drill Rig and Grader:** mapped in the [dataset manifest](../Dataset_Manifest/manifest.md); paired experiments not yet evaluated.

## How we add the next result

Run the new model in `Models/`, save its sensor and visual metrics under `Results/<equipment>/`, then add or update its entry in the `equipment` list above and one evidence line here. **Do not copy a score from another equipment mapping as independent evidence.** Dozer currently reuses Belt Conveyor's same 16 CORA runs; Wheel Loader shares some CORA healthy/bearing runs with Excavator. Only Pump currently exceeds 80% in both modalities on the strict test.